# Interactive Sanger Sequencing Simulation

Use the dropdowns to adjust parameters and see how they affect the sequencing gel.

In [ ]:
import random
import pandas as pd
import altair as alt

# Allow larger datasets
alt.data_transformers.disable_max_rows()

In [ ]:
def ddN(number_of_iterations, template, base, ddN_ratio):
    """Simulate ddNTP termination for a single nucleotide type."""
    new_strands = []
    for _ in range(number_of_iterations):
        synthesized_strand = ''
        for nucleotide in template:
            if nucleotide == base and random.random() > ddN_ratio:
                synthesized_strand += nucleotide.lower()
                break
            synthesized_strand += nucleotide
        new_strands.append(synthesized_strand)
    return new_strands

In [ ]:
# Pre-generate data for multiple parameter combinations
all_data = []

for seq_length in [100, 200, 300, 400, 500]:
    seq = ''.join(random.choice('ATCG') for _ in range(seq_length))
    
    for ddN_ratio in [0.7, 0.8, 0.9, 0.95]:
        for iterations in [1000, 5000, 10000]:
            
            # Run simulation
            for nt in 'ATCG':
                strands = ddN(iterations, seq, nt, ddN_ratio)
                for strand in strands:
                    all_data.append({
                        'seq_length': seq_length,
                        'ddN_ratio': ddN_ratio,
                        'iterations': iterations,
                        'base': nt,
                        'length': len(strand)
                    })

df = pd.DataFrame(all_data)
print(f"Generated {len(df)} data points")

In [ ]:
# Group by parameters and compute counts
df_grouped = df.groupby(
    ['seq_length', 'ddN_ratio', 'iterations', 'base', 'length']
).size().reset_index(name='count')

print(f"Grouped to {len(df_grouped)} unique combinations")

## Interactive Gel Visualization

Use the dropdown menus to select parameters:
- **Sequence Length**: Length of template DNA (bp)
- **ddN Ratio**: Higher = longer fragments (more dNTP vs ddNTP)
- **Iterations**: Number of polymerase molecules

In [ ]:
# Create interactive selections
seq_length_select = alt.binding_select(options=[100, 200, 300, 400, 500], name='Seq Length: ')
seq_length_param = alt.param('seq_length_param', value=300, bind=seq_length_select)

ddN_ratio_select = alt.binding_select(options=[0.7, 0.8, 0.9, 0.95], name='ddN Ratio: ')
ddN_ratio_param = alt.param('ddN_ratio_param', value=0.9, bind=ddN_ratio_select)

iterations_select = alt.binding_select(options=[1000, 5000, 10000], name='Iterations: ')
iterations_param = alt.param('iterations_param', value=5000, bind=iterations_select)

# Build chart with filtering
chart = alt.Chart(df_grouped).mark_tick(thickness=4).encode(
    y=alt.Y('length:Q', scale=alt.Scale(type='sqrt'), title='Fragment Length'),
    x=alt.X('base:N', title='Nucleotide'),
    color=alt.Color('count:Q', legend=None,
                    scale=alt.Scale(type='log', scheme='greys')),
    tooltip=['base:N', 'length:Q', 'count:Q']
).transform_filter(
    (alt.datum.seq_length == seq_length_param) &
    (alt.datum.ddN_ratio == ddN_ratio_param) &
    (alt.datum.iterations == iterations_param)
).add_params(
    seq_length_param,
    ddN_ratio_param,
    iterations_param
).properties(
    width=200,
    height=600,
    title='Sanger Sequencing Gel'
)

chart